# Sentiment Analysis: Training & Evaluation

This notebook trains a sentiment classifier on Twitter airline tweets.

**Workflow:**
1. Load and explore data
2. Text preprocessing
3. Split train/val/test
4. Train TF-IDF + LogisticRegression
5. Evaluate and analyze
6. Save model artifacts

**Dataset:** 14,640 airline tweets (negative, neutral, positive)

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

import importlib
import sentiment_utils
importlib.reload(sentiment_utils)

from sentiment_utils import (
    load_data,
    preprocess_dataframe,
    split_data,
    vectorize_and_train,
    evaluate_model,
    get_feature_importance,
    LABEL_MAP,
)

from joblib import dump

DATA_PATH = "data/Tweets.csv"
VECT_PATH = "tfidf_vectorizer.joblib"
MODEL_PATH = "logreg_sentiment_model.joblib"

print("✓ Imports OK")

## Load Data

In [ ]:
df_raw = load_data(DATA_PATH)
print(f"Shape: {df_raw.shape}")
print(f"\nFirst few rows:")
print(df_raw.head())

## EDA with YData Profiling

In [ ]:
from ydata_profiling import ProfileReport

print("Generating profiling report...")
profile = ProfileReport(
    df_raw,
    title="Airline Sentiment - EDA",
    minimal=False
)
profile.to_file("airline_sentiment_profile.html")
print("✓ Report saved to airline_sentiment_profile.html")

## Raw Distribution

In [ ]:
raw_counts = df_raw['airline_sentiment'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d32f2f', '#f57c00', '#388e3c']
raw_counts.plot(kind='bar', color=colors, ax=ax)
ax.set_title('Label Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

for i, v in enumerate(raw_counts):
    ax.text(i, v + 100, f'{v}\n({v/len(df_raw)*100:.1f}%)', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nClass Imbalance:")
print(f"  Negative: {raw_counts['negative']/len(df_raw)*100:.1f}%")
print(f"  Neutral:  {raw_counts['neutral']/len(df_raw)*100:.1f}%")
print(f"  Positive: {raw_counts['positive']/len(df_raw)*100:.1f}%")
print(f"  Ratio (neg/pos): {raw_counts['negative']/raw_counts['positive']:.1f}x")

## Preprocess

In [ ]:
df_clean = preprocess_dataframe(df_raw)

print(f"\nCleaned shape: {df_clean.shape}")
print(f"\nSample cleaning:")
for idx in range(3):
    print(f"\n[Original]")
    print(f"  {df_clean.iloc[idx]['text'][:100]}...")
    print(f"[Cleaned]")
    print(f"  {df_clean.iloc[idx]['clean_text']}")

## Text Length Analysis

In [ ]:
df_clean['text_len'] = df_clean['clean_text'].str.len()
df_clean['word_count'] = df_clean['clean_text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_clean['text_len'].hist(bins=50, ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Character Length', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Length')
axes[0].set_ylabel('Count')

df_clean['word_count'].hist(bins=50, ax=axes[1], color='lightcoral', edgecolor='black')
axes[1].set_title('Word Count', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Words')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"\nStats:")
print(f"  Avg length: {df_clean['text_len'].mean():.0f} chars")
print(f"  Avg words: {df_clean['word_count'].mean():.1f}")

## Split Data

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df_clean)

print(f"\nClass distribution in splits:")
for split_name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    print(f"\n{split_name}:")
    for label_id in range(3):
        count = (y_split == label_id).sum()
        pct = count / len(y_split) * 100
        print(f"  {LABEL_MAP[label_id]:10s}: {count:5d} ({pct:5.1f}%)")

## Train Model

In [ ]:
vectorizer, model = vectorize_and_train(X_train, y_train)

## Evaluate

In [ ]:
val_metrics = evaluate_model(vectorizer, model, X_val, y_val, "Validation")

In [ ]:
test_metrics = evaluate_model(vectorizer, model, X_test, y_test, "Test")

## Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = test_metrics['confusion_matrix']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=[LABEL_MAP[i] for i in range(3)],
            yticklabels=[LABEL_MAP[i] for i in range(3)])
axes[0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True')
axes[0].set_xlabel('Predicted')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', ax=axes[1],
            xticklabels=[LABEL_MAP[i] for i in range(3)],
            yticklabels=[LABEL_MAP[i] for i in range(3)])
axes[1].set_title('Normalized', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

## Error Analysis

In [ ]:
X_test_vec = vectorizer.transform(X_test)
y_pred = model.predict(X_test_vec)

errors = y_test != y_pred
error_count = errors.sum()
total = len(y_test)

print(f"\nMisclassifications: {error_count}/{total} ({error_count/total*100:.1f}%)")

print(f"\nBreakdown by true label:")
for true_id in range(3):
    true_label = LABEL_MAP[true_id]
    mask = (y_test == true_id) & errors
    if mask.sum() > 0:
        print(f"\n  When true={true_label}:")
        for pred_id in range(3):
            pred_label = LABEL_MAP[pred_id]
            count = ((y_test == true_id) & (y_pred == pred_id) & errors).sum()
            if count > 0:
                print(f"    Predicted {pred_label}: {count}x")

# Show a few examples
error_idx = np.where(errors)[0]
if len(error_idx) > 0:
    print(f"\nExample errors:")
    for i, idx in enumerate(error_idx[:3]):
        true_label = LABEL_MAP[int(y_test.iloc[idx])]
        pred_label = LABEL_MAP[int(y_pred[idx])]
        text = X_test.iloc[idx]
        
        print(f"\n{i+1}. Text: {text[:80]}...")
        print(f"   True: {true_label}, Pred: {pred_label}")

## Feature Importance

In [ ]:
importance = get_feature_importance(vectorizer, model, top_n=15)

print("\nTop words per class:")
for label, features in importance.items():
    print(f"\n{label.upper()}:")
    for i, (word, coef) in enumerate(list(features.items())[:10], 1):
        print(f"  {i:2d}. {word:15s} ({coef:6.3f})")

## Visualize Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors_map = {'negative': '#d32f2f', 'neutral': '#f57c00', 'positive': '#388e3c'}

for idx, (label, features) in enumerate(importance.items()):
    words = list(features.keys())[:10]
    coefs = list(features.values())[:10]
    
    axes[idx].barh(words, coefs, color=colors_map[label])
    axes[idx].set_title(f'{label.upper()}', fontweight='bold')
    axes[idx].set_xlabel('Coefficient')
    axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

## Save Artifacts

In [ ]:
dump(vectorizer, VECT_PATH)
dump(model, MODEL_PATH)

print(f"✓ Vectorizer -> {VECT_PATH}")
print(f"✓ Model -> {MODEL_PATH}")
print(f"\n✓ Ready for inference!")

## Summary

In [ ]:
print(f"""
╔════════════════════════════════════════╗
║  TRAINING COMPLETE                     ║
╚════════════════════════════════════════╝

Dataset: 14,640 airline tweets
Classes: negative (63%), neutral (21%), positive (16%)

Model: Logistic Regression + TF-IDF
- Vocab size: 20,000
- N-grams: unigrams + bigrams
- Class weights: balanced

Results (Test Set):
- Accuracy:  {test_metrics['accuracy']:.3f}
- Precision: {test_metrics['precision_macro']:.3f}
- Recall:    {test_metrics['recall_macro']:.3f}
- F1-Score:  {test_metrics['f1_macro']:.3f}

Artifacts saved:
✓ {VECT_PATH}
✓ {MODEL_PATH}
✓ airline_sentiment_profile.html

Next: Open sentiment.API.ipynb for inference
""")